In [1]:
from nichenetpy.prediction import LigandActivityPredictor, LigandReceptorNetwork
from nichenetpy.utils import read_matrix_from_csv, read_csv_cols
from nichenetpy.extraction import get_expressed_genes

import anndata

In [2]:
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/testargs/ligand_target_matrix.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/csv/lr_network.csv")
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
ann

AnnData object with n_obs × n_vars = 5027 × 13541
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nGene', 'nUMI', 'aggregate', 'res.0.6', 'celltype'
    var: 'gene'
    layers: 'counts', 'data', 'scale.data'

In [3]:
celltype_counts = ann.obs["celltype"].value_counts()
celltype_counts

celltype
CD4 T    2562
CD8 T    1645
B         382
Treg      199
NK        131
Mono       90
DC         18
Name: count, dtype: int64

In [ ]:
receiver = "CD8 T"
expressed_genes_receiver = set(get_expressed_genes(receiver, ann, 0.05))
all_receptors = lr_network.get_receptors()
expressed_receptors = all_receptors.intersection(expressed_genes_receiver)
potential_ligands = set(key for key, group in lr_network.item_iter() if len(group.intersection(expressed_receptors)) > 0)

3903
1084
107
475


In [5]:
ligand_activities = predictor.predict_ligand_activities(
    geneset,
    background_expressed_genes,
    potential_ligands
)
ligand_activities = sorted(ligand_activities.items(), key=lambda x : x[1]["aupr_corrected"], reverse=True)
ligand_activities[:10]

[('"Ifna1"',
  {'aupr': np.float64(0.4178628711859948),
   'aupr_corrected': np.float64(0.3423379573658757)}),
 ('"Ifnl3"',
  {'aupr': np.float64(0.391939644541973),
   'aupr_corrected': np.float64(0.3164147307218539)}),
 ('"Ifnb1"',
  {'aupr': np.float64(0.38720798251986854),
   'aupr_corrected': np.float64(0.3116830686997495)}),
 ('"Il27"',
  {'aupr': np.float64(0.37815297574454027),
   'aupr_corrected': np.float64(0.30262806192442115)}),
 ('"Ifng"',
  {'aupr': np.float64(0.3655665422088432),
   'aupr_corrected': np.float64(0.29004162838872416)}),
 ('"Ifnk"',
  {'aupr': np.float64(0.27522591434367766),
   'aupr_corrected': np.float64(0.19970100052355858)}),
 ('"Ifne"',
  {'aupr': np.float64(0.27370092284997777),
   'aupr_corrected': np.float64(0.19817600902985869)}),
 ('"Ebi3"',
  {'aupr': np.float64(0.2545082226394273),
   'aupr_corrected': np.float64(0.1789833088193082)}),
 ('"Ifnl2"',
  {'aupr': np.float64(0.24461164764885127),
   'aupr_corrected': np.float64(0.16908673382873218)}